# পাঠ ১৩ - কগনি জ্ঞান গ্রাফ সহ এজেন্ট মেমোরি


## সেটআপ

এই নোটবুকটি দেখায় কিভাবে [**Cognee**](https://www.cognee.ai/) জ্ঞান গ্রাফ এবং **মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক** (MAF) ব্যবহার করে একটি বুদ্ধিমান **কোডিং সহকারী** স্থায়ী মেমরির সাথে তৈরি করা যায়।

Cognee অসমগঠিত টেক্সটকে একটি গঠিত, অনুসন্ধানযোগ্য জ্ঞান গ্রাফে রূপান্তর করে যা ভেক্টর এমবেডিং দ্বারা সমর্থিত — আপনার এজেন্টকে একটি সমৃদ্ধ, সম্পর্ক সচেতন দীর্ঘমেয়াদী মেমরি প্রদান করে।

### আপনি যা শিখবেন
1. **জ্ঞান গ্রাফ তৈরি করুন**: ডেভেলপার প্রোফাইল এবং সেরা অনুশীলনকে গঠিত, অনুসন্ধানযোগ্য জ্ঞানে রূপান্তর করুন।
2. **Cognee কে MAF এর সাথে একীভূত করুন**: `@tool` ফাংশন ব্যবহার করে একটি MAF এজেন্টকে Cognee এর জ্ঞান গ্রাফ থেকে প্রশ্ন করার সুযোগ দিন।
3. **সেশন সচেতন কথোপকথন**: একই সেশনের মধ্যে একাধিক প্রশ্নের প্রসঙ্গ বজায় রাখুন।
4. **দীর্ঘমেয়াদী মেমরি**: সেশন জুড়ে গুরুত্বপূর্ণ জ্ঞান সংরক্ষণ করুন এবং নতুন কথোপকথনে তা পুনরুদ্ধার করুন।

### প্রয়োজনীয়তা
- পাইথন ৩.৯+
- লোকালহোস্টে Redis চালু থাকা (`docker run -d -p 6379:6379 redis`) সেশন ম্যানেজমেন্টের জন্য
- একটি LLM API কী (যেমন OpenAI) — `.env` ফাইলে `LLM_API_KEY` সেট করুন
- `.env` ফাইলে `CACHING=true` (Cognee সেশনগুলোর জন্য প্রয়োজন)
- একটি Azure AI Foundry প্রকল্প এবং একটি ডিপ্লয়মেন্টকৃত চ্যাট মডেল
- `.env` ফাইলে `AZURE_AI_PROJECT_ENDPOINT` এবং `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI তে লগইন করা (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## এজেন্ট মেমোরির ধরন

এই নোটবুকটি মূল লেসন ১৩ নোটবুকের সাথে একই তিনটি মেমোরি ধরন পরীক্ষা করে, তবে লং-টার্ম মেমোরি ব্যাকএন্ড হিসেবে Cognee ব্যবহার করে:

| মেমোরির ধরন | পদ্ধতি | জীবনকাল |
|---|---|---|
| **ওয়ার্কিং** | `agent.create_session()` (MAF) | একক কথোপকথন |
| **শর্ট-টার্ম** | Cognee সেশন ক্যাশ (Redis) | একক সেশন |
| **লং-টার্ম** | Cognee জ্ঞান গ্রাফ + ভেক্টরস | স্থায়ী |

### Cognee এর মেমোরি স্থাপত্য
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## কগনী স্টোরেজ প্রস্তুত করুন


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — নলেজ বেস তৈরি

আমরা আমাদের কোডিং সহায়কের জন্য একটি ব্যাপক নলেজ বেস তৈরি করতে তিন ধরনের ডেটা গ্রহণ করি:

1. **ডেভেলপার প্রোফাইল** — ব্যক্তিগত দক্ষতা এবং প্রযুক্তিগত পটভূমি  
2. **পাইথন সেরা অনুশীলন** — প্রায়োগিক নির্দেশিকাযুক্ত পাইথনের জেন  
3. **ঐতিহাসিক আলোচনা** — ডেভেলপার এবং AI সহায়কদের মধ্যে অতীতের প্রশ্নোত্তর সেশনগুলি


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## জ্ঞান গ্রাফ দৃশ্যমান করুন

Cognee এটি বের করা সত্তা এবং সম্পর্কগুলির একটি ইন্টারেক্টিভ HTML ভিজ্যুয়ালাইজেশন রেন্ডার করতে পারে।


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify এর মাধ্যমে মেমরি সমৃদ্ধ করুন

`memify()` জ্ঞান গ্রাফ বিশ্লেষণ করে এবং বুদ্ধিমান নিয়ম তৈরি করে — নিদর্শন, সেরা চর্চা, এবং ধারণাগুলোর মধ্যে সম্পর্ক চিহ্নিত করে।


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## অংশ ২ — কগনী টুলস সহ এমএএফ এজেন্ট

এখন আমরা একটি এমএএফ এজেন্ট তৈরি করব যা `@tool` ফাংশনের মাধ্যমে কগনী এর জ্ঞান গ্রাফকে কুইরি করতে পারে। এটি এজেন্টটিকে সেশনগুলোর মাধ্যমে কথোপকথনের প্রেক্ষাপট বজায় রেখে গ্রাফ-সচেতন সেমান্টিক সার্চের সম্পূর্ণ ক্ষমতা কাজে লাগাতে দেয়।


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## সেশনসহ ওয়ার্কিং মেমোরি

`AgentSession` (যা `agent.create_session()` এর মাধ্যমে তৈরি করা হয়) একটি সেশনের মধ্যে ওয়ার্কিং মেমোরি প্রদান করে। এজেন্ট পূর্বের বার্তাগুলির দিকে ফেরত দেখতে পারে এবং একই সাথে Cognee এর দীর্ঘমেয়াদি জ্ঞান গ্রাফও অনুসন্ধান করতে পারে।


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## নতুন সেশন — দীর্ঘমেয়াদী মেমোরি বজায় থাকে

নতুন একটি সেশন শুরু করলে ওয়ার্কিং মেমোরি সাফ হয়ে যায়, তবে Cognee জ্ঞান গ্রাফ এখনও উপলব্ধ থাকে। এজেন্ট সম্পূর্ণ নতুন কথোপকথনে একই দীর্ঘমেয়াদী জ্ঞান পুনরায় আনতে পারে।


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## সারাংশ

এই নোটবুকটিতে আপনি একটি কোডিং সহকারী তৈরি করেছেন যা **MAF-এর ওয়ার্কিং মেমরি** (`agent.create_session()`) কে **Cognee-এর দীর্ঘমেয়াদী জ্ঞান গ্রাফের** সাথে সংযুক্ত করে।

### আপনি যা শিখেছেন
1. **জ্ঞান গ্রাফ নির্মাণ**: Cognee অগঠিত টেক্সট গ্রহণ করে এবং একটি গ্রাফ + ভেক্টর মেমরি তৈরি করে।
2. **মেমিফাই দিয়ে গ্রাফ সমৃদ্ধকরণ**: বিদ্যমান গ্রাফের উপরে আহৃত তথ্য এবং সমৃদ্ধ সম্পর্ক তৈরি।
3. **MAF + Cognee সংযোগ**: `@tool` ফাংশনগুলো MAF এজেন্টকে Cognee-এর গ্রাফ প্রাকৃতিকভাবে অনুসন্ধান করতে দেয়।
4. **ওয়ার্কিং মেমরি + দীর্ঘমেয়াদী মেমরি**: `AgentSession` (মাধ্যমে `agent.create_session()`) সেশন কন্টেক্সট দেয়, যখন Cognee অবিচ্ছিন্ন জ্ঞান প্রদান করে।
5. **NodeSets দিয়ে ফিল্টার করা অনুসন্ধান**: নির্দিষ্ট জ্ঞানের গ্রাফের অংশ লক্ষ্য করুন (যেমন শুধুমাত্র নীতিমালা)।

### মূল উপসংহার
- **Cognee** কাঁচা টেক্সটকে কাঠামোগত, সম্পর্ক সচেতন মেমরিতে রূপান্তর করে — যা সাধারণ ভেক্টর স্টোর থেকে শক্তিশালী।
- **`@tool` ফাংশন**গুলো MAF এজেন্ট এবং বাহ্যিক জ্ঞান সিস্টেমের মধ্যে পরিষ্কার সংযোগ স্থাপন করে।
- **`AgentSession`** (মাধ্যমে `agent.create_session()`) প্রতিটি কথোপকথনের প্রসঙ্গ দীর্ঘস্থায়ী জ্ঞান থেকে পৃথক রাখে।
- একই জ্ঞান গ্রাফ একাধিক সেশন এবং এজেন্টকে সেবা দেয়।

### বাস্তব জীবনের প্রয়োগ
- **ডেভেলপার কোপাইলট**: কোড রিভিউ, ইস্যু বিশ্লেষণ, আর্কিটেকচার সহায়ক
- **গ্রাহক-সামনে কোপাইলট**: পণ্য ডকুমেন্টেশন, FAQ, এবং CRM নোটের ওপর সহায়ক
- **অভ্যন্তরীণ বিশেষজ্ঞ কোপাইলট**: নীতি, আইনগত বা নিরাপত্তা সহায়ক যাঁরা নির্দেশিকার ওপর যুক্তি দেয়
- **এককৃত ডেটা স্তর**: কাঠামোগত এবং অগঠিত ডেটাকে একত্রিত করে একটি অনুসন্ধানযোগ্য গ্রাফে পরিণত করা

### পরবর্তী ধাপ
- Cognee-তে কালানুক্রমিক সচেতনতা নিয়ে পরীক্ষা-নিরীক্ষা করা
- ডোমেইন-বিশেষ গ্রাফ গুণগতমানের জন্য একটি OWL অন্টোলজি সংজ্ঞায়িত করা
- পুনরুদ্ধার প্রক্রিয়া উন্নত করার জন্য ব্যবহারকারী ফিডব্যাক চক্র যোগ করা
- একই Cognee মেমরি স্তর ভাগ করে নেওয়া মাল্টি-এজেন্ট সিস্টেমে স্কেল করা


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**বাতিল ঘোষণা**:  
এই ডকুমেন্টটি এআই অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অসঙ্গতির সম্ভাবনা থাকতে পারে। মূল ডকুমেন্টটি তার নিজস্ব ভাষায়ই কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ গ্রহণ করাই প্রয়োজনীয়। এই অনুবাদের ব্যবহার থেকে সৃষ্ট কোনো ভুল ধারণা বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
